## 1. Imports & Configuration


In [1]:
import os
import sys
import pandas as pd
import numpy as np
import warnings
import joblib

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import RobustScaler
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

warnings.filterwarnings('ignore')

# Print working directory and verify paths explicitly as requested
EXPERIMENTS_DIR = os.getcwd()
DATA_FILE = os.path.join(EXPERIMENTS_DIR, "data", "train_data.csv")
MODELS_DIR = os.path.join(EXPERIMENTS_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"Working Directory: {EXPERIMENTS_DIR}")
print(f"Dataset Path: {DATA_FILE}")
print(f"Models Export Path: {MODELS_DIR}")

Working Directory: e:\Projects\stockai-pro\experiments
Dataset Path: e:\Projects\stockai-pro\experiments\data\train_data.csv
Models Export Path: e:\Projects\stockai-pro\experiments\models


## 2. Load Data


In [2]:
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(f"Dataset missing! Could not find: {DATA_FILE}")

# Engine Python is used to avoid parsing warnings
df = pd.read_csv(DATA_FILE)
print(f"Successfully loaded dataset: {len(df)} rows | {df.shape[1]} columns.")
display(df.tail(3))

Successfully loaded dataset: 113222 rows | 14 columns.


,ticker,date,open,high,low,close,volume,ema_20,ema_50,rsi_14,macd,macd_signal,vwap,target
113219,BAJAJFINSV,2026-03-18 09:00:00,1791.800049,1794.000000,1791.000000,1791.900024,22025.0,1788.212222,1779.499041,60.877521,5.216288,5.499252,1960.030233,1
113220,BAJAJFINSV,2026-03-18 09:15:00,1791.500000,1794.500000,1790.099976,1793.699951,22237.0,1788.734863,1780.055940,64.050993,5.026255,5.404652,1959.975507,0
113221,BAJAJFINSV,2026-03-18 09:30:00,1795.000000,1796.699951,1788.599976,1790.099976,115327.0,1788.864873,1780.449823,54.932271,4.532912,5.230304,1959.690619,0


## 3. Data Cleaning & Base Feature Validation


In [3]:
# Handle missing values and normalize structures
df.columns = [str(c).lower().strip() for c in df.columns]

if 'close' in df.columns and 'target' in df.columns:
    df.dropna(subset=['close', 'target'], inplace=True)
else:
    raise ValueError("Dataset missing vital 'close' or 'target' target columns!")

# Ensure required features exist natively
necessary_cols = ['open', 'high', 'low', 'close', 'volume']
for req in necessary_cols:
    if req not in df.columns:
        raise ValueError(f"CRITICAL: Missing required numeric column '{req}'")

# Force conversions to float where appropriate
for col in necessary_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(subset=necessary_cols, inplace=True)

# Recreate Base Indicators if missing
if 'ema_20' not in df.columns: df['ema_20'] = df['close'].ewm(span=20).mean()
if 'ema_50' not in df.columns: df['ema_50'] = df['close'].ewm(span=50).mean()
if 'macd' not in df.columns: df['macd'] = df['close'].ewm(span=12).mean() - df['close'].ewm(span=26).mean()
if 'macd_hist' not in df.columns: df['macd_hist'] = df['macd'] - df['macd'].ewm(span=9).mean()
if 'vwap' not in df.columns:
    # Volume weighted average price approximation
    df['vwap'] = (df['close'] * df['volume']).cumsum() / df['volume'].cumsum()

print("Base features validated securely.")

Base features validated securely.


## 4. Advanced Feature Engineering\nAdd: `price_change`, `volatility`, `momentum`, `rolling_mean_5`, `rolling_std_5` per ticker


In [4]:
def engineer_advanced_features(group):
    group = group.copy()
    c = group['close']
    h = group['high']
    l = group['low']
    
    group['price_change'] = c - group['open']
    
    # Avoid zero-division with clip
    safe_c = c.clip(lower=1e-5)
    group['volatility'] = (h - l) / safe_c
    group['momentum'] = c - c.shift(3)
    group['rolling_mean_5'] = c.rolling(window=5, min_periods=1).mean()
    group['rolling_std_5'] = c.rolling(window=5, min_periods=2).std()
    
    return group

# Use modern group mapping
new_df = []
for ticker, group in df.groupby('ticker'):
    new_df.append(engineer_advanced_features(group))
    
df = pd.concat(new_df).reset_index(drop=True)

# Thoroughly wipe generated infinity/NaN from shifting calculations
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(0, inplace=True)

print(f"Data remaining after engineering: {len(df)} rows ready for modeling.")

Data remaining after engineering: 113222 rows ready for modeling.


## 5. Time Series Train/Test Split\nProper sequential split (`shuffle=False`)


In [5]:
# Strict Time-bound sorting to eliminate data leakage
if 'date' in df.columns:
    df.sort_values(by='date', inplace=True)
df.reset_index(drop=True, inplace=True)

# Exclude identifier/target columns
exclude_cols = ['ticker', 'date', 'target']
feature_cols = [c for c in df.columns if c not in exclude_cols and np.issubdtype(df[c].dtype, np.number)]
print(f"Selected {len(feature_cols)} robust predictors.")

X = df[feature_cols].values
y = df['target'].values

# Split exactly chronologically 80% Train, 20% Test (shuffle=False implicit to slicing)
split_idx = int(len(X) * 0.80)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Training samples: {len(X_train)}")
print(f"Testing samples : {len(X_test)}")

# Safely Scale utilizing mathematical resilience
scaler = RobustScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

X_train_sc = np.nan_to_num(X_train_sc, nan=0.0)
X_test_sc  = np.nan_to_num(X_test_sc, nan=0.0)

Selected 17 robust predictors.
Training samples: 90577
Testing samples : 22645


## 6. Model Training\nTrain XGBoost, RandomForest, and LogisticRegression natively.


In [6]:
models = {
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.05, use_label_encoder=False, eval_metric='logloss'),
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=6, min_samples_split=10, n_jobs=-1),
    'LogisticRegression': LogisticRegression(max_iter=500, class_weight='balanced')
}

predictions = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_sc, y_train)
    predictions[name] = model.predict(X_test_sc)

print("All algorithms optimized and mapped.")

Training XGBoost...
Training RandomForest...
Training LogisticRegression...
All algorithms optimized and mapped.


## 7. Model Comparison\nCalculate Accuracy, Precision, Recall, and F1 Score.


In [7]:
results = []

for name, preds in predictions.items():
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1_Score': f1
    })

results_df = pd.DataFrame(results).set_index('Model')
display(results_df.round(4))

,Accuracy,Precision,Recall,F1_Score
Model,,,,
XGBoost,0.5364,0.5026,0.1481,0.2288
RandomForest,0.5370,0.5280,0.0260,0.0495
LogisticRegression,0.5068,0.4721,0.5254,0.4973


## 8. Select & Save Best Model\nPick the winner natively and write `model.pkl` to `experiments/models/`


In [8]:
# Automatically choose the model with the highest structural F1 Score
best_model_name = results_df['F1_Score'].idxmax()
best_model = models[best_model_name]

print(f"\\n\u2605 BEST PERFORMER: {best_model_name} (F1 Score: {results_df.loc[best_model_name, 'F1_Score']:.4f})")

# Export to exactly the requested path: experiments/models/model.pkl
model_path = os.path.join(MODELS_DIR, "model.pkl")

# We highly recommend dumping the scaler and features vector list for API bindings as well
scaler_path = os.path.join(MODELS_DIR, "scaler.pkl")
features_path = os.path.join(MODELS_DIR, "features.pkl")

joblib.dump(best_model, model_path)
joblib.dump(scaler, scaler_path)
joblib.dump(feature_cols, features_path)

print(f"\\nSUCCESS! Model accurately saved securely to:")
print(f" -> {model_path}")
print("Ready for live trading deployment.")

\n★ BEST PERFORMER: LogisticRegression (F1 Score: 0.4973)
\nSUCCESS! Model accurately saved securely to:
 -> e:\Projects\stockai-pro\experiments\models\model.pkl
Ready for live trading deployment.
